# C1.1 · Agentic offensive workflow

**Function C — Offensive Security & Research → The Pentester / Red Teamer**  ·  *AI for Security*

---

**Risk.** Payload suggestions instead of attack chains.

**Control.** Feed the harness full target context before it swings.

**This lab.** Drive a planner/executor pair against a local target you own.

| | |
|---|---|
| Open-source tooling | CAI, Metasploit, InterCode-CTF |
| Open-weight models | Kimi K2, GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C1.1"))

An agentic offensive workflow is the same loop as everything else in B2 — the difference is that the *target* is not yours, so the stop condition and the containment are the professional obligations.

In [ ]:
from cybercommons import loop, sandbox

# scope is a control, not a paragraph in the engagement letter
IN_SCOPE = {"target.example", "api.target.example"}
box = sandbox.Sandbox(
    egress=sandbox.EgressPolicy(allow_hosts=IN_SCOPE),
    paths=sandbox.PathGuard(workspace="/work/engagement"),
    tools=sandbox.ToolPolicy(allow={"http_get", "read_file"},
                             require_approval={"write_file"},
                             deny={"exploit_write", "lateral_move"}))

for target in ["https://target.example/login",
               "https://api.target.example/v1/users",
               "https://unrelated.example/",          # out of scope
               "http://169.254.169.254/"]:
    print(box.call("http_get", target))

The out-of-scope host is refused by the same mechanism that refuses the metadata service. Scope enforcement that lives only in the operator's attention is not enforcement — and an agent has no attention at all.

In [ ]:
trace = loop.run(loop.FakeModel(["enumerate endpoints",
                                 "enumerate endpoints",
                                 "found /v1/users"]),
                 loop.unit_test(lambda s: s.startswith("found"), "found something"),
                 goal="enumerate the API", max_steps=5)
print(trace.table())

### Expect

The two in-scope hosts are allowed; the unrelated host and the metadata address are denied. The loop stops after three steps when the verifier is satisfied.

### Your turn

Add a rate limit to the sandbox and decide its unit. For an engagement, requests-per-second protects the client's production service — which is part of the scope agreement, not a nicety.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C1.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*